# Zagros-Makran benchmark: pipeline

This notebook runs the six analysis steps in order. Each step is also
executable as a standalone script from the command line.

Before running, place the raw ISC Bulletin export at `data/Final_.csv`
or set `RAW_FILENAME` in `zm/config.py`.


## Environment


In [ ]:
import sys, subprocess, platform, os

print('python    ', sys.version.split()[0], platform.system())
for pkg in ('numpy', 'pandas', 'scipy', 'sklearn', 'matplotlib', 'tensorflow'):
    try:
        mod = __import__(pkg)
        print('{:10s}'.format(pkg), getattr(mod, '__version__', 'unknown'))
    except ImportError:
        print('{:10s}'.format(pkg), 'not installed')


In [ ]:
import zm.config as C

print(C.summary())
print('base directory:', C.BASE_DIR)


## Running the steps

Each cell runs one step in a separate process and streams its output.
Step 2 takes several hours for the full configuration; it resumes from
existing weight files, so it can be interrupted and restarted.

Set `ZM_QUICK=1` in the environment for a short test run with two seeds
and eight epochs.


In [ ]:
def run(script):
    print('running', script, flush=True)
    p = subprocess.Popen([sys.executable, script], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    print('exit code', p.returncode, flush=True)
    return p.returncode


### Step 0: catalog construction


In [ ]:
run('run_00_clean.py')


### Step 1: completeness, b-value and splits


In [ ]:
run('run_01_catalog.py')


### Step 2: training


In [ ]:
run('run_02_train.py')


### Step 3: baselines, ETAS and inference


In [ ]:
run('run_03_eval.py')


### Step 4: Integrated Gradients


In [ ]:
run('run_04_ig.py')


### Step 5: figures


In [ ]:
run('run_05_figures.py')


### Step 6: collected quantities


In [ ]:
run('run_06_numbers.py')


## Results

All reported quantities are collected in `outputs/NUMBERS.md`, and every
plotted value in `outputs/figures/figure_values.json`.


In [ ]:
for root, dirs, files in os.walk('outputs'):
    dirs[:] = [d for d in dirs if d != 'weights']
    for f in sorted(files):
        p = os.path.join(root, f)
        print('{:56s} {:>10d}'.format(p, os.path.getsize(p)))
